In [14]:
import os
from dotenv import load_dotenv
print(load_dotenv())


True


In [15]:
API_KEY = os.getenv("API_KEY")

In [16]:
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = API_KEY

llm = init_chat_model(
    "llama-3.3-70b-versatile", 
    model_provider="groq",
    temperature=0
)

In [17]:
from langchain_community.document_loaders import PyPDFLoader

loader1 = PyPDFLoader("contract1.pdf")
loader2 = PyPDFLoader("contract2.pdf")

docs1 = loader1.load()
docs2 = loader2.load()

# Add source tagging
for doc in docs1:
    doc.metadata["source"] = "Contract A"

for doc in docs2:
    doc.metadata["source"] = "Contract B"

documents = docs1 + docs2

In [41]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = splitter.split_documents(documents)
docs

[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-03-03T18:00:22+05:30', 'author': 'varshini ramesh', 'moddate': '2026-03-03T18:00:22+05:30', 'source': 'Contract A', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='NON – DISCLOSURE AGREEMENT  \n This Non-Disclosure Agreement ("Agreement") is entered into as of \nDD/MM/2025, by and between:  \nName: M/s. Bisleri Pvt Ltd  \nAddress: 86, Collector Sivakumar street, KK Pudhur, Coimbatore IN ("Referrer")  \nAND  \nName: [Product Company Name]  \nAddress: [Product Company Address] ("Solution Provider") \nNOW, THIS AGREEMENT SHALL WITNESSETH AS FOLLOWS:  \n  \n1. Term  \nThis Agreement shall remain in force for a period of ten (10) years from the Effective Date \nand shall automatically renew for successive two (2) year periods unless either party \nprovides written notice of non-renewal at least hundred and twenty (120) days prior to the \nexpiration of the the curr

In [40]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

# Example
texts = ["This is contract A clause", "This is contract B clause"]

embeddings = model.encode(texts)
embeddings

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

array([[-3.60938832e-02,  1.04321659e-01, -1.22655947e-02,
        -1.18197512e-03, -4.57374193e-02,  1.05475634e-01,
         9.37454030e-02,  1.83690321e-02,  5.74798808e-02,
         3.39886248e-02,  4.15212562e-04, -1.37151573e-02,
         3.30684446e-02,  9.34713054e-03,  2.85855830e-02,
         4.27583465e-03, -1.56967323e-02,  1.75687373e-02,
        -4.77291197e-02,  1.38169294e-02,  3.64498496e-02,
         9.94266570e-02, -5.97754233e-02, -1.07629411e-02,
         3.38767096e-02, -1.25253052e-02,  4.80115376e-02,
         3.38873826e-02,  5.34664914e-02, -2.45842133e-02,
        -1.00283213e-01,  1.01722650e-01, -5.66123659e-03,
         2.70120259e-02,  5.87520301e-02,  1.97601933e-02,
        -4.94241863e-02, -8.91224518e-02, -3.92048471e-02,
         9.64822806e-03,  3.21129411e-02, -3.75878923e-02,
        -1.63323712e-02,  7.61251943e-03, -6.10497827e-03,
         4.11916189e-02, -3.58817317e-02,  4.04032283e-02,
        -2.21289117e-02,  4.21265028e-02, -2.83068437e-0

In [25]:
from sentence_transformers import SentenceTransformer

class CustomEmbeddings:
    def __init__(self):
        self.model = model
    
    def embed_documents(self, texts):
        return self.model.encode(texts).tolist()
    
    def embed_query(self, text):
        return self.model.encode([text])[0].tolist()
    
    def __call__(self, text):
        return self.embed_query(text)

In [26]:
from langchain_community.vectorstores import FAISS

embeddings = CustomEmbeddings()

vectorstore = FAISS.from_documents(docs, embeddings)

retriever = vectorstore.as_retriever()

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


In [22]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [31]:
query = "Compare risks between Termination clause of Contract A and Contract B"

In [32]:
relevant_docs = retriever.invoke(query)
relevant_docs

[Document(id='7988cb6d-e866-4ce6-a228-e90927655da1', metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-03-03T18:00:22+05:30', 'author': 'varshini ramesh', 'moddate': '2026-03-03T18:00:22+05:30', 'source': 'Contract A', 'total_pages': 3, 'page': 1, 'page_label': '2'}, page_content='3. Limitation of Liability  \nIn no event shall our party be liable to the other for any indirect, incidental, consequential, \nspecial, or punitive damages arising out of or relating to this Agreement.  \nThe total aggregate liability of each party under this Agreement shall be limited to INR \n1,00,000 (Rupees One Lakh Only).  \nNothing in this clause shall limit liability arising from gross negligence, willful misconduct, or \nbreach of confidentiality obligations.  \n  \n4. Termination  \nEither party may terminate this Agreement by providing thirty (30) days written notice to \nthe other party.  \nUpon termination, each party shall, upon written reques

In [35]:
context = "\n\n".join([
    f"{doc.metadata.get('source', 'Unknown')}:\n{doc.page_content}"
    for doc in relevant_docs
])
context

'Contract A:\n3. Limitation of Liability  \nIn no event shall our party be liable to the other for any indirect, incidental, consequential, \nspecial, or punitive damages arising out of or relating to this Agreement.  \nThe total aggregate liability of each party under this Agreement shall be limited to INR \n1,00,000 (Rupees One Lakh Only).  \nNothing in this clause shall limit liability arising from gross negligence, willful misconduct, or \nbreach of confidentiality obligations.  \n  \n4. Termination  \nEither party may terminate this Agreement by providing thirty (30) days written notice to \nthe other party.  \nUpon termination, each party shall, upon written request, return or destroy all confidential \ninformation within fifteen (15) days.  \nThe confidentiality obligations set forth herein shall survive termination for a period of two  \n(2) year from the effective date of termination.  \n  \n5. General  \nThis Agreement contains the entire agreement between the parties and sup

In [39]:
prompt = f"""
You are a legal expert.

Compare Contract A and Contract B based on the context.

Identify:
- Risks
- Conflicts
- Liability issues
- Which contract is riskier and why

Context:
{context}

Question:
{query}
"""

response = llm.invoke(prompt)

print(response.content)

After analyzing the Termination clauses of Contract A and Contract B, here's a comparison of the risks:

**Similarities:**

1. Both contracts allow either party to terminate the agreement with 30 days' written notice.
2. Both contracts require the parties to maintain confidentiality and security of confidential information after termination.

**Differences:**

1. **Survival of Confidentiality Obligations:** Contract A specifies that the confidentiality obligations will survive termination for a period of 2 years from the effective date of termination. In contrast, Contract B states that the obligations of confidentiality will survive the termination of the agreement, but it does not specify a specific time period. However, it mentions that the confidentiality obligations will continue for a period of 2 years from the date of the last disclosure by the Company.
2. **Return or Destruction of Confidential Information:** Contract A requires each party to return or destroy all confidential 